Install PyNeuraLogic from PyPI

In [15]:
# ! pip install neuralogic

In [16]:
# import logging
# import sys

# logging.basicConfig(format='%(asctime)s | %(levelname)s : %(message)s', level=logging.INFO, stream=sys.stdout)

### Faster Data Download using Rsync
Instead of looping through hundreds of files with individual `scp` calls, we can use `rsync` to sync the entire remote directory to Colab in one session. We'll use `sshpass` to handle the password non-interactively.

In [17]:
# !apt-get install -y sshpass

# import os
# import subprocess

# SSH_USER_HOST = "kassyask@login3.rci.cvut.cz"
# SSH_PASSWORD = ""
# REMOTE_BASE = "/home/kassyask/neuro-vector-symbolic-architectures-raven/data/I-RAVEN-10000/"
# LOCAL_DIR = "/content/sample_data/raven_data/"

# os.makedirs(LOCAL_DIR, exist_ok=True)

# CONSTELLATIONS = [
#     "center_single", "distribute_four", "distribute_nine",
#     "in_center_single_out_center_single", "in_distribute_four_out_center_single",
#     "left_center_single_right_center_single", "up_center_single_down_center_single"
# ]

# MAX_TRAIN_INSTANCES = 200
# MAX_TEST_INSTANCES = 20

# def run_command(cmd_str, input_data=None):
#     process = subprocess.run(cmd_str, shell=True, capture_output=True, text=True, input=input_data)
#     return process.stdout.splitlines(), process.stderr.splitlines(), process.returncode

# print(f"Starting limited download to {LOCAL_DIR}")

# for constellation in CONSTELLATIONS:
#     remote_constellation_path = os.path.join(REMOTE_BASE, constellation)
#     local_constellation_path = os.path.join(LOCAL_DIR, constellation)
#     os.makedirs(local_constellation_path, exist_ok=True)

#     print(f"\nProcessing constellation: {constellation}")

#     # List remote files
#     list_cmd = f"sshpass -p '{SSH_PASSWORD}' ssh -o StrictHostKeyChecking=no {SSH_USER_HOST} 'ls {remote_constellation_path}'"
#     stdout, stderr, _ = run_command(list_cmd)

#     all_remote_files = [f.strip() for f in stdout if f.strip()]

#     # Organize by base name
#     train_files = [f for f in all_remote_files if "_train." in f]
#     test_files = [f for f in all_remote_files if "_test." in f]

#     def select_files(file_list, limit):
#         bases = sorted(list(set(f.replace(".npz", "").replace(".xml", "") for f in file_list)))
#         selected = []
#         for b in bases[:limit]:
#             if f"{b}.npz" in file_list: selected.append(f"{b}.npz")
#             if f"{b}.xml" in file_list: selected.append(f"{b}.xml")
#         return selected

#     to_download = select_files(train_files, MAX_TRAIN_INSTANCES) + select_files(test_files, MAX_TEST_INSTANCES)

#     if to_download:
#         print(f"  Downloading {len(to_download)} files...")
#         files_input = "\n".join(to_download) + "\n"

#         # Corrected rsync command structure
#         rsync_cmd = (
#             f"sshpass -p '{SSH_PASSWORD}' rsync -avz --files-from=- "
#             f"-e 'ssh -o StrictHostKeyChecking=no' "
#             f"{SSH_USER_HOST}:{remote_constellation_path}/ {local_constellation_path}/"
#         )

#         stdout_r, stderr_r, rc = run_command(rsync_cmd, input_data=files_input)
#         if rc == 0:
#             print("  Success.")
#         else:
#             print(f"  Error (RC {rc}):")
#             for line in stderr_r: print(f"    {line}")
#     else:
#         print("  No files found.")

# print("\nDownload process finished.")

Vizualize the example

In [18]:
# import numpy as np
# from PIL import Image, ImageDraw, ImageFont
# import xml.etree.ElementTree as ET

# # Define necessary variables for this cell
# save_path = "/content/I-RAVEN_samples/png.png"
# sample_data_path = "/content/sample_data/" # Ensure this is defined
# sample_data_prefix = f"{sample_data_path}/RAVEN_0_train"
# tree = ET.parse(f"{sample_data_prefix}.xml")
# root = tree.getroot()
# # Assuming a default config for visualization if not explicitly passed from elsewhere
# config = "in_center_single_out_center_single"
# pos_map = {
#     "center_single": {0: 0},
#     "distribute_four": {0: 1, 1: 2, 2: 3, 3: 4},
#     "distribute_nine": {0: 5, 1: 6, 2: 7, 3: 8, 4: 9, 5: 10, 6: 11, 7: 12, 8: 13},
#     "left_center_single_right_center_single": {0: 14, 1: 15},
#     "up_center_single_down_center_single": {0: 16, 1: 17},
#     "in_center_single_out_center_single": {0: 0, 1: 9},
#     "in_distribute_four_out_center_single": {0: 0, 1: 18, 2: 19, 3: 20, 4: 21},
# }

# def _get_font(size):
#     try:
#         return ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", size)
#     except:
#         return ImageFont.load_default()

# # Re-adjusting the shape mapping based on user visual feedback
# # Current evidence:
# # - Objects drawn as pentagons are labeled as circles
# # - Objects drawn as hexagons are labeled as pentagons
# # - Objects drawn as circles are labeled as hexagons
# TYPE_NAMES = {0: "tri", 1: "sq", 2: "pent", 3: "hex", 4: "circ"}

# def panel_matches(pred, gt):
#     if len(pred) != len(gt): return False
#     p_sorted = sorted(pred, key=lambda x: x['position'])
#     g_sorted = sorted(gt, key=lambda x: x['position'])
#     for p, g in zip(p_sorted, g_sorted):
#         if p != g: return False
#     return True

# def parse_visual_data(root, config):
#     images_data = np.load(f"{sample_data_path}/RAVEN_0_train.npz")
#     images = images_data['image']
#     target = int(root.attrib.get('target', images_data.get('target', 0)))

#     all_panels_attributes = []
#     panels = root[0]

#     for panel in panels:
#         objs = []
#         global_offset = 0
#         for struct in panel:
#             for comp in struct:
#                 for layout in comp:
#                     pos_list = eval(layout.attrib["Position"])
#                     for entity in layout:
#                         bbox = eval(entity.attrib["bbox"])
#                         local_pos = pos_list.index(bbox) + global_offset
#                         objs.append({
#                             "type": int(entity.attrib["Type"]) - 1,
#                             "size": int(entity.attrib["Size"]),
#                             "color": int(entity.attrib["Color"]),
#                             "position": pos_map[config][local_pos]
#                         })
#                     global_offset += len(pos_list)
#         all_panels_attributes.append(objs)

#     return images, target, all_panels_attributes

# def visualize_sample(images, target, pred_panels, gt_panels, save_path, config=None):
#     H, W = images.shape[1], images.shape[2]
#     pad, ann_h, border = 6, 60, 3
#     font, font_sm = _get_font(11), _get_font(10)
#     cell_w, cell_h = W + pad, H + ann_h + pad

#     ctx_cols, ctx_rows = 3, 3
#     ctx_w, ctx_h = ctx_cols * cell_w - pad, ctx_rows * cell_h - pad
#     ans_cols, ans_w, ans_h = 8, 8 * cell_w - pad, cell_h

#     total_w = max(ctx_w, ans_w) + 2 * pad
#     gap_between, title_h = 16, 22
#     total_h = title_h + ctx_h + gap_between + ans_h + pad

#     canvas = Image.new("RGB", (total_w, total_h), (255, 255, 255))
#     draw = ImageDraw.Draw(canvas)
#     draw.text((pad, 2), f"Config: {config or 'unknown'}   |   Correct answer: candidate {target}", fill=(60, 60, 60), font=font)

#     def _paste_panel(panel_idx, x0, y0, highlight=False):
#         panel_img = Image.fromarray(images[panel_idx]).convert("RGB")
#         if highlight:
#             bordered = Image.new("RGB", (W + 2*border, H + 2*border), (0, 180, 0))
#             bordered.paste(panel_img, (border, border))
#             canvas.paste(bordered, (x0 - border, y0 - border))
#         else:
#             canvas.paste(panel_img, (x0, y0))

#         pred = pred_panels[panel_idx]
#         gt = gt_panels[panel_idx] if gt_panels else None
#         match = panel_matches(pred, gt) if gt else None

#         lines = [f"p{obj['position']} {TYPE_NAMES.get(obj['type'], 't'+str(obj['type']))} s{obj['size']} c{obj['color']}" for obj in pred] or ["[empty]"]
#         color = (0, 140, 0) if match else (200, 0, 0) if match is not None else (60, 60, 60)

#         ty = y0 + H + 2
#         for line in lines[:4]:
#             draw.text((x0, ty), line, fill=color, font=font_sm)
#             ty += 14

#     ctx_x0, ctx_y0 = (total_w - ctx_w) // 2, title_h
#     for row in range(ctx_rows):
#         for col in range(ctx_cols):
#             idx = row * 3 + col
#             x, y = ctx_x0 + col * cell_w, ctx_y0 + row * cell_h
#             if idx < 8: _paste_panel(idx, x, y)
#             else:
#                 qimg = Image.new("RGB", (W, H), (180, 180, 180))
#                 ImageDraw.Draw(qimg).text((W//2-6, H//2-8), "?", fill=(80, 80, 80), font=_get_font(24))
#                 canvas.paste(qimg, (x, y))

#     ans_x0, ans_y0 = (total_w - ans_w) // 2, ctx_y0 + ctx_h + gap_between
#     for i in range(8): _paste_panel(8 + i, ans_x0 + i * cell_w, ans_y0, highlight=(i == target))

#     canvas.save(save_path)
#     display(canvas)

# imgs, tgt, attrs = parse_visual_data(root, config)
# visualize_sample(imgs, tgt, attrs, attrs, "raven_sample.png", config=config)

### 1. Parsing the Example and Building the Dataset

Imports & Constants


In [19]:
import xml.etree.ElementTree as ET
import numpy as np
import os, time

import neuralogic
neuralogic.set_max_memory_size(32)

from neuralogic.core import R, V, Template, Settings
from neuralogic.dataset import Dataset
from neuralogic.nn import get_evaluator
from neuralogic.optim import Adam
from neuralogic.nn.loss import MSE
from tqdm.auto import tqdm

# ── Constants ──────────────────────────────────────────────────────────────────
CONSTELLATIONS = [
    "center_single",
    "distribute_four",
    "distribute_nine",
    "in_center_single_out_center_single",
    "in_distribute_four_out_center_single",
    "left_center_single_right_center_single",
    "up_center_single_down_center_single",
]
LOCAL_DATA_DIR = "/home/kassyask/neuro-vector-symbolic-architectures-raven/data/I-RAVEN-10000"

NUM_CONTEXT    = 8
NUM_CANDIDATES = 8
NUM_POSITIONS  = 22
TYPE_RANGE     = range(1, 6)
SIZE_RANGE     = range(0, 7)
COLOR_RANGE    = range(0, 10)
EMBED_DIM      = 8
MAX_TRAIN      = 500
MAX_TEST       = 50
EPOCHS         = 200
LOG_EVERY      = 20

POS_MAP = {
    "center_single":                          {0: 0},
    "distribute_four":                        {0: 1, 1: 2, 2: 3, 3: 4},
    "distribute_nine":                        {i: i + 5 for i in range(9)},
    "left_center_single_right_center_single": {0: 14, 1: 15},
    "up_center_single_down_center_single":    {0: 16, 1: 17},
    "in_center_single_out_center_single":     {0: 0,  1: 9},
    "in_distribute_four_out_center_single":   {0: 0, 1: 18, 2: 19, 3: 20, 4: 21},
}

PANEL_TO_COORD = {
    0: (0,0), 1: (0,1), 2: (0,2),
    3: (1,0), 4: (1,1), 5: (1,2),
    6: (2,0), 7: (2,1),
}
for _c in range(NUM_CANDIDATES):
    PANEL_TO_COORD[NUM_CONTEXT + _c] = (2, 2)

PANEL_SPACING = 10.0
POS_CENTERS   = {i: (float(i % 6), float(i // 6)) for i in range(NUM_POSITIONS)}
SPATIAL_RELS  = [
    "left_of_intra", "right_of_intra", "above_intra", "below_intra",
    "left_of_inter", "right_of_inter", "above_inter", "below_inter",
]

print("Constants ready ✓")

Constants ready ✓


Parsing & Sub-graph Construction

In [20]:

# ── Spatial helpers ────────────────────────────────────────────────────────────
def _global_xy(panel_idx, pos):
    r, c = PANEL_TO_COORD[panel_idx]
    lx, ly = POS_CENTERS.get(pos, (0.0, 0.0))
    return c * PANEL_SPACING + lx, r * PANEL_SPACING + ly

def _spatial_rels_intra(sx, sy, dx, dy, eps=1e-4):
    rels = []
    if   sx < dx - eps: rels.append("left_of_intra")
    elif sx > dx + eps: rels.append("right_of_intra")
    if   sy < dy - eps: rels.append("above_intra")
    elif sy > dy + eps: rels.append("below_intra")
    return rels

def _spatial_rels_inter(sx, sy, dx, dy, eps=1e-4):
    rels = []
    if   sx < dx - eps: rels.append("left_of_inter")
    elif sx > dx + eps: rels.append("right_of_inter")
    if   sy < dy - eps: rels.append("above_inter")
    elif sy > dy + eps: rels.append("below_inter")
    return rels

def are_adjacent(p, q):
    r1, c1 = PANEL_TO_COORD[p]
    r2, c2 = PANEL_TO_COORD[q]
    return abs(r1 - r2) + abs(c1 - c2) == 1

print("Spatial helpers ready ✓")


# ── Parsing ────────────────────────────────────────────────────────────────────
def parse_panel(panel_elem, panel_idx, config_name):
    objects, obj_count = [], 0
    for struct in panel_elem:
        for comp in struct:
            for layout in comp:
                pos_list = eval(layout.attrib["Position"])
                for entity in layout:
                    obj_id     = f"p{panel_idx}_o{obj_count}"
                    local_idx  = pos_list.index(eval(entity.attrib["bbox"]))
                    mapped_pos = POS_MAP.get(config_name, {}).get(local_idx, 0)
                    objects.append((
                        obj_id, mapped_pos,
                        int(entity.attrib["Type"]),
                        int(entity.attrib["Size"]),
                        int(entity.attrib["Color"]),
                    ))
                    obj_count += 1
    return objects


def build_subgraph(panel_objects_list, cand_idx):
    facts      = []
    cand_panel = NUM_CONTEXT + cand_idx
    active     = list(range(NUM_CONTEXT)) + [cand_panel]
    panel_nodes = {}
    edge_counter = [0]

    def _add_edges(ni, nj, rels):
        for rel in rels:
            eid = f"e{edge_counter[0]}"
            edge_counter[0] += 1
            facts.append(R.edge(ni, nj, eid))
            facts.append(R.get(rel)(eid))

    for panel_idx in active:
        nodes = []
        for obj_id, pos, typ, sz, col in panel_objects_list[panel_idx]:
            facts.append(R.get(f"type_{typ}")(obj_id))
            facts.append(R.get(f"size_{sz}")(obj_id))
            facts.append(R.get(f"color_{col}")(obj_id))
            facts.append(R.get(f"pos_{pos}")(obj_id))
            nodes.append((obj_id, pos))
        panel_nodes[panel_idx] = nodes

    # Intra-panel edges
    for panel_idx in active:
        nodes = panel_nodes[panel_idx]
        for i, (ni, pi) in enumerate(nodes):
            for j, (nj, pj) in enumerate(nodes):
                if i == j:
                    continue
                sx, sy = POS_CENTERS[pi]
                dx, dy = POS_CENTERS[pj]
                _add_edges(ni, nj, _spatial_rels_intra(sx, sy, dx, dy))

    # Inter-panel edges (adjacent panels only)
    for p in active:
        for q in active:
            if p == q or not are_adjacent(p, q):
                continue
            for ni, pi in panel_nodes[p]:
                gxi, gyi = _global_xy(p, pi)
                for nj, pj in panel_nodes[q]:
                    gxj, gyj = _global_xy(q, pj)
                    _add_edges(ni, nj, _spatial_rels_inter(gxi, gyi, gxj, gyj))

    for obj_id, _ in panel_nodes[cand_panel]:
        facts.append(R.is_cand(obj_id))

    return facts

print("Parsing helpers ready ✓")

Spatial helpers ready ✓
Parsing helpers ready ✓


Build Datasets

In [ ]:

# ── Dataset loading ────────────────────────────────────────────────────────────
def load_rpms(split, max_problems):
    """Returns list of ([subgraph_facts × 8], target) per RPM."""
    rpms = []
    for constellation in tqdm(CONSTELLATIONS, desc=f"Loading {split}"):
        folder = os.path.join(LOCAL_DATA_DIR, constellation)
        if not os.path.exists(folder):
            continue
        xml_files = sorted(
            f for f in os.listdir(folder) if f.endswith(f"_{split}.xml")
        )[:max_problems]
        for f_xml in xml_files:
            base     = f_xml.replace(".xml", "")
            xml_path = os.path.join(folder, f_xml)
            npz_path = os.path.join(folder, base + ".npz")
            if not os.path.isfile(npz_path):
                continue
            tree   = ET.parse(xml_path)
            target = int(np.load(npz_path)["target"])
            panel_objects_list = [
                parse_panel(p, i, constellation)
                for i, p in enumerate(tree.getroot()[0])
            ]
            rpms.append((
                [build_subgraph(panel_objects_list, cid) for cid in range(NUM_CANDIDATES)],
                target,
            ))
    return rpms


def rpms_to_dataset(rpms, oversample_correct=1):
    """Flat NeuraLogic dataset: label=1.0 for correct candidate, 0.0 otherwise."""
    ds = Dataset()
    for subgraph_facts_list, target in rpms:
        for cid in range(NUM_CANDIDATES):
            label   = 1.0 if cid == target else 0.0
            repeats = oversample_correct if cid == target else 1
            for _ in range(repeats):
                ds.add_example(subgraph_facts_list[cid])
                ds.add_query(R.predict[label])
    return ds


print("Loading training data...")
t0 = time.time()
rpms_train = load_rpms("train", MAX_TRAIN)
print(f"  {len(rpms_train)} RPMs in {time.time()-t0:.1f}s\n")

print("Loading test data...")
t0 = time.time()
rpms_test_by_config = {}
for constellation in tqdm(CONSTELLATIONS, desc="Test sets"):
    folder = os.path.join(LOCAL_DATA_DIR, constellation)
    if not os.path.exists(folder):
        continue
    xml_files = sorted(
        f for f in os.listdir(folder) if f.endswith("_test.xml")
    )[:MAX_TEST]
    targets, rpms = [], []
    for f_xml in xml_files:
        base     = f_xml.replace(".xml", "")
        xml_path = os.path.join(folder, f_xml)
        npz_path = os.path.join(folder, base + ".npz")
        if not os.path.isfile(npz_path):
            continue
        tree   = ET.parse(xml_path)
        target = int(np.load(npz_path)["target"])
        targets.append(target)
        panel_objects_list = [
            parse_panel(p, i, constellation)
            for i, p in enumerate(tree.getroot()[0])
        ]
        rpms.append((
            [build_subgraph(panel_objects_list, cid) for cid in range(NUM_CANDIDATES)],
            target,
        ))
    rpms_test_by_config[constellation] = (targets, rpms)
print(f"  Done in {time.time()-t0:.1f}s")


Loading training data...


Loading train:   0%|          | 0/7 [00:00<?, ?it/s]

Template

In [ ]:

# ── Template ───────────────────────────────────────────────────────────────────
def build_template(D=EMBED_DIM):
    template = Template()

    template.add_rules([R.type_emb(V.X)[D,]  <= R.get(f"type_{t}")(V.X)  for t in TYPE_RANGE])
    template.add_rules([R.size_emb(V.X)[D,]  <= R.get(f"size_{s}")(V.X)  for s in SIZE_RANGE])
    template.add_rules([R.color_emb(V.X)[D,] <= R.get(f"color_{c}")(V.X) for c in COLOR_RANGE])
    template.add_rules([R.pos_emb(V.X)[D,]   <= R.get(f"pos_{p}")(V.X)   for p in range(NUM_POSITIONS)])
    template.add_rules([R.edge_emb(V.E)[D,]  <= R.get(rel)(V.E)           for rel in SPATIAL_RELS])

    template += R.h0(V.X) <= (R.type_emb(V.X)[D, D], R.size_emb(V.X)[D, D],
                               R.color_emb(V.X)[D, D], R.pos_emb(V.X)[D, D])

    def gnn_layer(h_out, h_in):
        nonlocal template # Add this line
        template += R.get(h_out)(V.X) <= (
            R.get(h_in)(V.Y)[D, D],
            R.edge(V.X, V.Y, V.E),
            R.edge_emb(V.E),
        )

    gnn_layer("h1", "h0")
    gnn_layer("h2", "h1")

    template += R.predict[1, D] <= (R.h2(V.X), R.is_cand(V.X))
    template += R.predict[1, D] <= (R.h0(V.X), R.is_cand(V.X))

    return template

template = build_template()
print("Template built ✓")


In [ ]:
# template.draw()

In [ ]:
print(template)

Ground

In [ ]:
# import traceback

# EPOCHS = 200
# LR     = 0.005
# settings  = Settings(optimizer=Adam(lr=LR), epochs=EPOCHS, error_function=CrossEntropy())
# evaluator = get_evaluator(template, settings)
# print("Grounding training dataset")
# t0 = time.time()

# try:
#     built_train = evaluator.build_dataset(train_ds)
# except Exception as e:
#     traceback.print_exc()

# print(f"  Grounded in {time.time()-t0:.1f}s\n")

Train

In [ ]:

# ── Training ───────────────────────────────────────────────────────────────────
settings    = Settings(optimizer=Adam(lr=5e-3), epochs=EPOCHS, error_function=MSE())
evaluator   = get_evaluator(template, settings)
train_ds    = rpms_to_dataset(rpms_train, oversample_correct=2)
built_train = evaluator.build_dataset(train_ds)

print(f"\nTraining for {EPOCHS} epochs...")
t0 = time.time()
for epoch, (loss, n) in enumerate(evaluator.train(built_train, epochs=EPOCHS), 1):
    if epoch % LOG_EVERY == 0 or epoch == EPOCHS:
        print(f"  Epoch {epoch:4d}/{EPOCHS}  loss = {loss/n:.4f}")
print(f"  Done in {time.time()-t0:.1f}s")




Evaluate

In [ ]:
# ── Evaluation — argmax over 8 candidate scores per RPM ───────────────────────
print("\nEvaluation:")
total_correct, total_rpms = 0, 0
for constellation, (targets, rpms) in rpms_test_by_config.items():
    test_ds    = rpms_to_dataset(rpms)
    built_test = evaluator.build_dataset(test_ds)
    scores     = list(evaluator.test(built_test))  # flat list, 8 scores per RPM

    correct = 0
    for rpm_idx, target in enumerate(targets):
        chunk     = scores[rpm_idx * NUM_CANDIDATES : (rpm_idx + 1) * NUM_CANDIDATES]
        predicted = int(np.argmax(chunk))
        if predicted == target:
            correct += 1

    acc = correct / len(targets)
    print(f"  {constellation}: {correct}/{len(targets)} = {acc:.3f}")
    total_correct += correct
    total_rpms    += len(targets)

print(f"\n  Overall: {total_correct}/{total_rpms} = {total_correct/total_rpms:.3f}")